# 🎨 Sketch2Shape AI - GPU Inference Server (Google Colab)

This notebook provides a cloud GPU backend for **Sketch2Shape AI** to run heavy 3D reconstruction models (TripoSR, Zero123, Stable Fast 3D, Shap-E) on free T4/A100 GPUs and expose an API endpoint for the local application.

### Workflow:
1. Verify GPU acceleration (`nvidia-smi`)
2. Clone official model repositories & install dependencies
3. Download pretrained checkpoints
4. Launch FastAPI server with public tunnel (ngrok or Cloudflared)
5. Copy public URL into the Sketch2Shape local web app

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install dependencies
!pip install -q fastapi uvicorn pyngrok trimesh pillow opencv-python-headless numpy scipy pydantic
# Clone TripoSR for ultra-fast 3D reconstruction from single image / sketch
!git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/tatsy/torchmcubes.git

In [ ]:
# Step 3: Load Model & Warm Up
import torch
from tsr.system import TSR
import trimesh
import numpy as np
from PIL import Image

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load TripoSR model checkpoint automatically from HuggingFace
model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(8192)
model.to(device)
print("✅ Model loaded successfully into GPU VRAM!")

In [ ]:
# Step 4: FastAPI Server Setup
import io
import base64
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
import nest_asyncio
import uvicorn

app = FastAPI(title="Sketch2Shape Colab GPU Engine")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class InferenceRequest(BaseModel):
    image_b64: str
    prompt: str = ""
    mc_resolution: int = 256

@app.get("/health")
def health():
    return {"status": "ready", "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}

@app.post("/api/generate-3d")
def generate_3d(req: InferenceRequest):
    try:
        img_data = req.image_b64
        if "," in img_data:
            img_data = img_data.split(",", 1)[1]
        img_bytes = base64.b64decode(img_data)
        input_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        
        with torch.no_grad():
            scene_codes = model(input_image, device=device)
            meshes = model.extract_mesh(scene_codes, resolution=req.mc_resolution)
            mesh = meshes[0]
            
        # Export to GLB format in memory
        glb_buffer = io.BytesIO()
        mesh.export(glb_buffer, file_type="glb")
        glb_bytes = glb_buffer.getvalue()
        
        return Response(content=glb_bytes, media_type="model/gltf-binary")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✅ FastAPI inference routes registered!")

In [ ]:
# Step 5: Start Tunnel and Server
# Get a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "" # Paste your token here if using ngrok

from pyngrok import ngrok
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

public_url = ngrok.connect(8000).public_url
print(f"\n🚀 SKETCH2SHAPE COLAB ENDPOINT URL:\n👉 {public_url} 👈\n")
print("Copy the above URL into your local Sketch2Shape app under 'Colab Server URL'.\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)